# RAG Tutorial — Full Pipeline

**Goal:** Build a working RAG system and understand each stage.

Follow the pipeline from PDF loading and chunking through embeddings, indexing, retrieval, generation, and testing.

Use the separate In-Class Exercise notebook for short experiments with chunk size, embeddings, `top_k`, and prompts.

# 1. Environment Setup

Use this environment to run the notebook consistently across classroom machines.

### Core Tools

- **Python / Jupyter:** notebook runtime
- **LlamaIndex:** RAG orchestration
- **Chroma:** local vector store
- **Qwen Embedding:** converts text to vectors
- **Qwen LLM:** generates answers from retrieved context
- **PyPDF:** extracts text from PDFs

In [1]:
# Check the Python runtime and environment
import sys
import platform
from pathlib import Path

print("=" * 60)
print("Environment Check")
print("=" * 60)

print(f"Python       : {sys.version}")
print(f"Executable   : {sys.executable}")
print(f"Platform     : {platform.platform()}")
print(f"Working dir  : {Path.cwd().resolve()}")

assert sys.version_info >= (3, 9), "Python >= 3.9 is required."

print("\n✅ Python version check passed.")

Environment Check
Python       : 3.11.16 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:36:16) [MSC v.1942 64 bit (AMD64)]
Executable   : D:\anaconda3\envs\tutorial\python.exe
Platform     : Windows-10-10.0.26200-SP0
Working dir  : D:\RAG\02_RAG tutorial

✅ Python version check passed.


## 1.1 Dependency Check

For an existing environment, install the requirements:

```powershell
pip install -r requirements.txt
```

Keeping installation outside the notebook avoids version changes, kernel mismatches, and restarts during class.

In [2]:
# Check required packages
import importlib

packages = {
    "llama_index": "LlamaIndex",
    "chromadb": "Chroma",
    "openai": "OpenAI SDK",
    "pypdf": "PyPDF",
    "dotenv": "python-dotenv",
    "pandas": "Pandas",
    "sklearn": "scikit-learn",
}

print("=" * 60)
print("Package Check")
print("=" * 60)

for module, name in packages.items():
    try:
        importlib.import_module(module)
        print(f"✅ {name}")
    except ImportError as e:
        print(f"❌ {name}: {e}")

Package Check
✅ LlamaIndex
✅ Chroma
✅ OpenAI SDK
✅ PyPDF
✅ python-dotenv
✅ Pandas
✅ scikit-learn


# 2. API Configuration

Keep the provider, model names, and API endpoint in one place. Switching providers should require configuration changes, not a rewrite of the RAG pipeline.

```text
DASHSCOPE_API_KEY=your_api_key
DASHSCOPE_BASE_URL=https://dashscope.aliyuncs.com/compatible-mode/v1
LLM_MODEL=qwen3.7-flash
EMBEDDING_MODEL=qwen3.7-text-embedding
```

Never add a real API key to the notebook or commit `.env` to GitHub.

> Model Studio API keys are region-bound. Use an API key created for the region of your Base URL.

In [3]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY") or os.getenv("dashscope_api_key")
DASHSCOPE_BASE_URL = (
    os.getenv("DASHSCOPE_BASE_URL")
    or os.getenv("dashscope_base_url")
    or "https://dashscope.aliyuncs.com/compatible-mode/v1"
)

LLM_MODEL = os.getenv("LLM_MODEL", "qwen3.7-flash")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "qwen3.7-text-embedding")

print("=" * 60)
print("Model Configuration")
print("=" * 60)

print(f"LLM model       : {LLM_MODEL}")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Base URL        : {DASHSCOPE_BASE_URL}")

if DASHSCOPE_API_KEY:
    print("API key         : ✅ loaded")
else:
    print("API key         : ❌ missing")

print("\n✅ Model configuration successful.")

print("\n✅ Session 2 complete: model configuration loaded.")


Model Configuration
LLM model       : qwen3.7-flash
Embedding model : qwen3.7-text-embedding
Base URL        : https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1
API key         : ✅ loaded

✅ Model configuration successful.

✅ Session 2 complete: model configuration loaded.


## 2.1 API Connectivity Test

Run both checks before continuing:

1. Confirm the chat model can answer a prompt.
2. Confirm the embedding model returns a vector.

If either check fails, verify the API key, endpoint, and model name.

In [4]:
from openai import OpenAI

assert DASHSCOPE_API_KEY, (
    "DASHSCOPE_API_KEY not found. "
    "Please configure your .env file first."
)

client = OpenAI(
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
)

# ---- Test LLM ----
response = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "user", "content": "Explain RAG in one sentence."}
    ],
)

print("LLM test:")
print(response.choices[0].message.content)

# ---- Test Embedding ----
embedding_response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input="This is a test sentence for generating an embedding."
)

embedding = embedding_response.data[0].embedding

print("\nEmbedding test:")
print(f"Vector dimension = {len(embedding)}")
print(f"First 10 values  = {embedding[:10]}")

LLM test:
RAG (Retrieval-Augmented Generation) is an AI framework that enhances large language models by dynamically retrieving relevant information from an external knowledge base and using that context to generate more accurate, grounded, and up-to-date responses.

Embedding test:
Vector dimension = 1024
First 10 values  = [0.023406831547617912, 0.011190440505743027, 0.014274620451033115, 0.007536290213465691, -0.001235729898326099, 0.009651517495512962, 0.048130929470062256, -0.0730070173740387, 0.017023149877786636, -0.024762095883488655]


# 3. Load Documents

Load source PDFs and inspect their structure before chunking.

## 3.0 Data Folders

```text
data/
├── raw/        source PDFs loaded by this notebook
├── processed/  optional cleaned or transformed files
├── eval/       optional question sets and reference answers
└── vector_db/  Chroma's persistent local database
```

Place the original PDFs in `data/raw/`. The `processed/` and `eval/` folders are reserved for future extensions; this notebook does not read them. Chroma creates the database in `vector_db/`, so do not place PDFs there.

In [5]:
from pathlib import Path

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EVAL_DIR = DATA_DIR / "eval"
VECTOR_DB_DIR = DATA_DIR / "vector_db"

for folder in [RAW_DIR, PROCESSED_DIR, EVAL_DIR, VECTOR_DB_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(RAW_DIR.glob("*.pdf"))

print("=" * 60)
print("Session 3 — PDF Inventory")
print("=" * 60)

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in {RAW_DIR.resolve()}. "
        "Please copy the course PDFs into data/raw/."
    )

for i, pdf in enumerate(pdf_files, start=1):
    print(f"{i:>2}. {pdf.name}  ({pdf.stat().st_size / 1024:.1f} KB)")

print(f"\nTotal PDFs: {len(pdf_files)}")
print("\n✅ Session 3 complete: data folders and PDF inventory ready.")


Session 3 — PDF Inventory
 1. ADV036_Pre-acceptance-requirements of MiC.pdf  (262.0 KB)

Total PDFs: 1

✅ Session 3 complete: data folders and PDF inventory ready.


In [6]:
from llama_index.core import SimpleDirectoryReader

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in {RAW_DIR.resolve()}.\n"
        "Please copy your course PDFs into data/raw/."
    )

reader = SimpleDirectoryReader(
    input_dir=str(RAW_DIR),
    # Set required_exts to restrict loading to specific file extensions.
    required_exts=[".pdf"],
    recursive=True,
)

documents = reader.load_data(show_progress=True)

print(f"\nLoaded document units: {len(documents)}")

Loading files:   0%|          | 0/1 [00:00<?, ?it/s]


Loaded document units: 15


## 3.1 Inspect the Loaded Documents

For PDFs, a `Document` often represents one page or parsed unit; it is not the final retrieval chunk. Preserve document-level metadata such as file names and page numbers so retrieved answers can cite their sources.

In [7]:
from IPython.display import display

# Inspect the first five document units
for i, doc in enumerate(documents[:5], start=1):
    print("=" * 80)
    print(f"Document unit #{i}")
    print(f"Metadata: {doc.metadata}")
    print(f"Text preview:\n{doc.text[:700]}")

Document unit #1
Metadata: {'page_label': '1', 'file_name': 'ADV036_Pre-acceptance-requirements of MiC.pdf', 'file_path': 'D:\\RAG\\02_RAG tutorial\\data\\raw\\ADV036_Pre-acceptance-requirements of MiC.pdf', 'file_type': 'application/pdf', 'file_size': 268275, 'creation_date': '2026-09-27', 'last_modified_date': '2026-09-17'}
Text preview:
Buildings Department 
Practice Note for Authorized Persons,  
Registered Structural Engineers and 
Registered Geotechnical Engineers 
           
ADV-36 
 
 
Modular Integrated Construction 
 
 
Introduction 
 
 Modular Integrated Construction (MiC) is a construction method that employs 
the technique of having freestanding volumetric modules (with finishes, fixtures, fittings, 
etc.) manufactured off -site and then transported to site for assembly .  P roven benefits 
include improved site safety, more efficient and better  quality control, shortened 
construction period, less construction waste , less demand for on- site labour , less 
disturbance 

In [8]:
# Run a basic data quality check
empty_docs = [i for i, doc in enumerate(documents) if not doc.text.strip()]
text_lengths = [len(doc.text) for doc in documents if doc.text.strip()]

print(f"Non-empty units : {len(text_lengths)}")
print(f"Empty units     : {len(empty_docs)}")

if text_lengths:
    print(f"Min chars      : {min(text_lengths):,}")
    print(f"Median chars   : {sorted(text_lengths)[len(text_lengths)//2]:,}")
    print(f"Max chars      : {max(text_lengths):,}")

Non-empty units : 15
Empty units     : 0
Min chars      : 776
Median chars   : 2,063
Max chars      : 3,015


# 4. Chunking

Split the loaded documents into manageable chunks for retrieval.


In [9]:
from llama_index.core.node_parser import SentenceSplitter

CHUNK_SIZE = 512
CHUNK_OVERLAP = 100

splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

nodes = splitter.get_nodes_from_documents(documents)

print("=" * 60)
print("Chunking Result")
print("=" * 60)

print(f"Original document units : {len(documents)}")
print(f"Generated chunks        : {len(nodes)}")
print(f"Chunk size              : {CHUNK_SIZE}")
print(f"Chunk overlap           : {CHUNK_OVERLAP}")

print("\n✅ Session 4 complete: chunks generated.")


Chunking Result
Original document units : 15
Generated chunks        : 23
Chunk size              : 512
Chunk overlap           : 100

✅ Session 4 complete: chunks generated.


# 5. Embedding

Convert each chunk into an embedding vector so that semantically related text can be retrieved by similarity search.


In [10]:
from llama_index.embeddings.openai import OpenAIEmbedding

embed_model = OpenAIEmbedding(
    model_name=EMBEDDING_MODEL,
    api_key=DASHSCOPE_API_KEY,
    api_base=DASHSCOPE_BASE_URL,
    embed_batch_size=20,
)

print(f"Embedding model ready: {EMBEDDING_MODEL}")
print("\n✅ Session 5 complete: embedding model configured.")


Embedding model ready: qwen3.7-text-embedding

✅ Session 5 complete: embedding model configured.


# 6. Create the Vector Index

Build a persistent index from the PDF chunks using **Chroma** for storage and **LlamaIndex** to connect the chunks, embedding model, and vector store.

To avoid duplicate records when rerunning the notebook, this section recreates the collection each time.

In [11]:
import chromadb
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

COLLECTION_NAME = "rag_course_demo"

chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_DB_DIR)
)

# Rebuild collection for a clean classroom run
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.get_or_create_collection(
    COLLECTION_NAME
)

vector_store = ChromaVectorStore(
    chroma_collection=collection
)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

print(f"Chroma collection: {COLLECTION_NAME}")
print(f"Persistent path  : {VECTOR_DB_DIR.resolve()}")

print("\n✅ Session 6 complete: vector index created.")


Chroma collection: rag_course_demo
Persistent path  : D:\RAG\02_RAG tutorial\data\vector_db

✅ Session 6 complete: vector index created.


In [12]:
# Build the index from the chunks we explicitly created above.
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True,
    
)

print("\n✅ Vector index created.")
print(f"Indexed chunks: {len(nodes)}")


Generating embeddings:   0%|          | 0/23 [00:00<?, ?it/s]

2026-09-28 10:09:15,134 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2026-09-28 10:09:15,736 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"



✅ Vector index created.
Indexed chunks: 23


## 6.1 Indexing Complete

```text
PDF → Documents → Chunks → Embeddings → Chroma
```

The knowledge base is indexed. A RAG query adds retrieval, context construction, and generation:

```text
Query → Retrieval → Augmentation → Generation
```

# 7. Retrieval

For each question, search the vector index and return the most relevant chunks.

In [13]:
TOP_K = 4

retriever = index.as_retriever(
    similarity_top_k=TOP_K
)

print(f"Retriever configured with top_k = {TOP_K}")

print("\n✅ Session 7 complete: retriever is ready.")


Retriever configured with top_k = 4

✅ Session 7 complete: retriever is ready.


In [14]:
import re
import html
from IPython.display import display, HTML

def get_file_name(node):
    metadata = node.metadata or {}
    return (
        metadata.get("file_name")
        or metadata.get("filename")
        or metadata.get("file_path")
        or "Unknown"
    )

def get_page_number(node):
    metadata = node.metadata or {}
    for key in ["page_label", "page_number", "page"]:
        value = metadata.get(key)
        if value is not None:
            try:
                return int(value)
            except (TypeError, ValueError):
                return value
    return None

def highlight_query_terms(text, query):
    """Highlight informative query terms in red."""
    stopwords = {
        "what", "is", "the", "a", "an", "of", "for", "on", "in", "to",
        "and", "each", "type", "be", "at", "as", "how", "many", "does",
        "will", "within", "according", "this", "document"
    }

    terms = [
        word.lower()
        for word in re.findall(r"\b[a-zA-Z0-9-]+\b", query)
        if word.lower() not in stopwords
    ]
    terms = sorted(set(terms), key=len, reverse=True)

    highlighted = html.escape(text)

    for term in terms:
        pattern = re.compile(
            rf"(?<![\w-])({re.escape(term)})(?![\w-])",
            re.IGNORECASE
        )
        highlighted = pattern.sub(
            r'<span style="color:#b00000; font-weight:bold;">\1</span>',
            highlighted
        )
    return highlighted

def inspect_retrieval(question, top_k=TOP_K, retrieved_nodes=None):

    # Run retrieval if results were not provided.
    if retrieved_nodes is None:
        retriever = index.as_retriever(
            similarity_top_k=top_k
        )
        retrieved_nodes = retriever.retrieve(question)

    print("=" * 80)
    print("RETRIEVED CHUNKS")
    print("=" * 80)

    for rank, item in enumerate(retrieved_nodes, start=1):

        node = item.node
        score = item.score

        file_name = get_file_name(node)
        page = get_page_number(node)

        print(f"\nRank {rank}")

        if score is not None:
            print(f"Score: {score:.4f}")
        else:
            print("Score: N/A")

        print(f"File : {file_name}")
        print(f"Page : {page}")

        print("\nChunk:")

        highlighted_text = highlight_query_terms(
            node.text,
            question
        )

        display(HTML(highlighted_text))

print("-" * 80)

print("\n✅ Retrieval inspection finished.")

--------------------------------------------------------------------------------

✅ Retrieval inspection finished.


In [29]:
question = (
    "What is the minimum sampling rate for on-site quality audit checks on each type of modular unit delivered to the building site?"
)

inspect_retrieval(question)

print("\n✅ Session 7 complete: retrieval results inspected.")


2026-09-22 16:17:51,600 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"


RETRIEVED CHUNKS

Rank 1
Score: 0.6679
File : ADV036_Pre-acceptance-requirements of MiC.pdf
Page : 9

Chunk:



Rank 2
Score: 0.5556
File : ADV036_Pre-acceptance-requirements of MiC.pdf
Page : 7

Chunk:



Rank 3
Score: 0.5290
File : ADV036_Pre-acceptance-requirements of MiC.pdf
Page : 8

Chunk:



Rank 4
Score: 0.4867
File : ADV036_Pre-acceptance-requirements of MiC.pdf
Page : 5

Chunk:



✅ Session 7 complete: retrieval results inspected.


## 7.1 Inspect Retrieval Results

Before moving on, ask students to check:

- Are all four returned chunks relevant?
- How do the first and fourth similarity scores compare?
- Do the chunks come from one PDF or several?
- Does a chunk contain matching terms but fail to answer the question?

Retrieval can fail before the LLM is involved. Poorly retrieved context makes a grounded answer unlikely.

# 8. Build the Complete RAG Query System

Connect retrieval, context construction, and generation into one pipeline.

In [15]:
from llama_index.core.llms import ChatMessage
from llama_index.llms.openai_like import OpenAILike

llm = OpenAILike(
    model=LLM_MODEL,
    api_key=DASHSCOPE_API_KEY,
    api_base=DASHSCOPE_BASE_URL,
    temperature=0.2,
    is_chat_model=True,
)

print(f"LLM ready: {LLM_MODEL}")

print("\n✅ Session 8 complete: LLM is ready.")


LLM ready: qwen3.7-flash

✅ Session 8 complete: LLM is ready.


In [16]:
SYSTEM_PROMPT = """You are a rigorous knowledge base question-answering assistant.

Please follow these rules:
1. Answer only based on the provided Context.
2. If the Context does not contain sufficient information to answer the question, explicitly state: “The knowledge base does not contain sufficient information.”
3. Do not add or infer facts that are not explicitly provided in the Context.
4. Keep your answers concise and clear whenever possible. 
5. At the end of your answer, provide the names of the source files used.
"""

def build_context(retrieved_nodes):
    context_blocks = []

    for i, item in enumerate(retrieved_nodes, start=1):
        node = item.node
        file_name = (
            node.metadata.get("file_name")
            or node.metadata.get("filename")
            or "unknown_file"
        )

        page = get_page_number(node)
        page_label = f", page {page}" if page is not None else ""

        context_blocks.append(
            f"[Source {i}] {file_name}{page_label}\n"
            f"{node.text}"
        )

    return "\n\n".join(context_blocks)


def build_rag_messages(question, retrieved_nodes):
    context = build_context(retrieved_nodes)

    user_prompt = f"""Please answer the following question.

Question:
{question}

Context:
{context}

Strictly base your answer on the provided Context.
"""

    return [
        ChatMessage(role="system", content=SYSTEM_PROMPT),
        ChatMessage(role="user", content=user_prompt),
    ]

def rag_query(question, top_k=TOP_K, verbose=True):

    # --------------------------------------------------
    # 1. Retrieval
    # --------------------------------------------------
    retriever = index.as_retriever(
        similarity_top_k=top_k
    )

    retrieved_nodes = retriever.retrieve(question)

    # --------------------------------------------------
    # 2. Build context
    # --------------------------------------------------
    messages = build_rag_messages(
        question,
        retrieved_nodes
    )

    # --------------------------------------------------
    # 3. Generate answer
    # --------------------------------------------------
    response = llm.chat(messages)

    answer = response.message.content

    # --------------------------------------------------
    # 4. Display
    # --------------------------------------------------
    if verbose:

        print("=" * 80)
        print("QUESTION")
        print("=" * 80)
        print(question)

        print("\nANSWER")
        print("=" * 80)
        print(answer)

        print("\nRETRIEVED CHUNKS")
        print("=" * 80)

        # Reuse retrieved_nodes to avoid running retrieval again.
        inspect_retrieval(
            question,
            retrieved_nodes=retrieved_nodes
        )

    # --------------------------------------------------
    # 5. Collect source information
    # --------------------------------------------------
    sources = []

    for rank, item in enumerate(retrieved_nodes, start=1):

        node = item.node

        sources.append({
            "rank": rank,
            "score": (
                float(item.score)
                if item.score is not None
                else None
            ),
            "file": get_file_name(node),
            "page": get_page_number(node),
            "node_id": node.node_id,
        })

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_nodes": retrieved_nodes,
    }

print("✅ Session 8 setup complete: RAG query function is ready.")

✅ Session 8 setup complete: RAG query function is ready.


## 8.1 Run a Complete RAG Query

Trace the data flow from question to answer:

```text
Question → Embedding → Retrieval → Context → Prompt → Qwen → Answer
```

In [32]:
demo_question = (
    "What is the minimum sampling rate for on-site quality audit checks on each type of modular unit delivered to the building site?"
)

result = rag_query(
    demo_question,
    top_k=4
)

print("\n✅ Session 8.1 complete: first RAG query finished.")

2026-09-22 16:20:16,727 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2026-09-22 16:20:27,705 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


QUESTION
What is the minimum sampling rate for on-site quality audit checks on each type of modular unit delivered to the building site?

ANSWER
Based on the provided context, the minimum sampling rate for on-site quality audit checks is at least 1% of each type of modular unit delivered to the building site.

Source files used: ADV036_Pre-acceptance-requirements of MiC.pdf

RETRIEVED CHUNKS
RETRIEVED CHUNKS

Rank 1
Score: 0.6680
File : ADV036_Pre-acceptance-requirements of MiC.pdf
Page : 9

Chunk:



Rank 2
Score: 0.5556
File : ADV036_Pre-acceptance-requirements of MiC.pdf
Page : 7

Chunk:



Rank 3
Score: 0.5291
File : ADV036_Pre-acceptance-requirements of MiC.pdf
Page : 8

Chunk:



Rank 4
Score: 0.4866
File : ADV036_Pre-acceptance-requirements of MiC.pdf
Page : 5

Chunk:



✅ Session 8.1 complete: first RAG query finished.


# 9. Testing

Testing checks whether the system runs and how it behaves. Try questions from four categories:

| Query type | What it tests |
|---|---|
| Direct fact | Basic retrieval |
| Concept | Semantic retrieval |
| Multi-sentence | Combining multiple chunks |
| Out of domain | Knowledge boundaries |

Replace the examples with questions grounded in your PDFs.

In [17]:
import pandas as pd

TEST_QUESTIONS = [
    # Use questions that exercise different retrieval and answer behaviors.
    "How many days does the Buildings Department (BD) take to provide a decision on a pre-submission enquiry?",
    "If the Buildings Department issues a letter of comments on a pre-acceptance application, within how many days must the Authorized Person (AP) and Registered Structural Engineer (RSE) clarify or respond to all comments before the application is considered abandoned?",
    "According to Table 1 of Appendix B, what is the minimum qualification required for a supervisor under the AP Stream, and how frequently must supervision be carried out?",
    "As an alternative arrangement to quality audits at the prefabrication factory, what is the minimum sampling rate for on-site quality audits of MiC modules delivered to the construction site?",
    "What qualification or certification must a factory have in order to manufacture Modular Integrated Construction (MiC) modules?",
]

test_results = []

for q in TEST_QUESTIONS:
    result = rag_query(q, top_k=TOP_K, verbose=False)

    test_results.append({
        "question": q,
        "answer": result["answer"],
        "source_count": len(result["sources"]),
        "top_source": result["sources"][0]["file"] if result["sources"] else None,
        "top_score": result["sources"][0]["score"] if result["sources"] else None,
    })

test_df = pd.DataFrame(test_results)

display(
    test_df.style.set_properties(
        subset=["question", "answer"],
        **{
            "white-space": "pre-wrap",
            "overflow-wrap": "anywhere",
            "min-width": "320px",
            "vertical-align": "top",
        },
    )
)

print("\n✅ Session 10 complete: basic testing finished.")

2026-09-28 10:09:36,909 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2026-09-28 10:09:45,479 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-28 10:09:47,410 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2026-09-28 10:10:01,539 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-28 10:10:02,287 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2026-09-28 10:10:26,417 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-28 10:10:27,405 - INFO - H

,question,answer,source_count,top_source,top_score
0,How many days does the Buildings Department (BD) take to provide a decision on a pre-submission enquiry?,"Based on the provided context, the Buildings Department (BD) will provide a determination on a pre-submission enquiry within **45 days**. **Source files used:** ADV036_Pre-acceptance-requirements of MiC.pdf",4,ADV036_Pre-acceptance-requirements of MiC.pdf,0.548267
1,"If the Buildings Department issues a letter of comments on a pre-acceptance application, within how many days must the Authorized Person (AP) and Registered Structural Engineer (RSE) clarify or respond to all comments before the application is considered abandoned?","Based on the provided context, the Authorized Person (AP) and Registered Structural Engineer (RSE) must clarify or respond to all comments within **30 days** from the date of the letter of comments. If they fail to do so, the application will be considered abandoned. Source file(s) used: ADV036_Pre-acceptance-requirements of MiC.pdf",4,ADV036_Pre-acceptance-requirements of MiC.pdf,0.659362
2,"According to Table 1 of Appendix B, what is the minimum qualification required for a supervisor under the AP Stream, and how frequently must supervision be carried out?","According to Table 1, the minimum qualification required for a supervisor under the AP Stream is Grade T3 (Technically Competent Person), and supervision must be carried out weekly. Source files used: ADV036_Pre-acceptance-requirements of MiC.pdf",4,ADV036_Pre-acceptance-requirements of MiC.pdf,0.494974
3,"As an alternative arrangement to quality audits at the prefabrication factory, what is the minimum sampling rate for on-site quality audits of MiC modules delivered to the construction site?","Based on the provided context, the minimum sampling rate for on-site quality audits of MiC modules delivered to the construction site is at least 1% of each type of modular unit delivered. Source files used: ADV036_Pre-acceptance-requirements of MiC.pdf",4,ADV036_Pre-acceptance-requirements of MiC.pdf,0.682998
4,What qualification or certification must a factory have in order to manufacture Modular Integrated Construction (MiC) modules?,"Based on the provided context, a prefabrication factory must hold one of the following qualifications/certifications: 1. A full certificate issued by the Building Technology Research Institute under the **Modular Integrated Construction Manufacturer Accreditation Scheme**; OR 2. **ISO 9001 quality assurance certification** for the prefabrication factory, combined with a Quality Assurance Scheme that complies with paragraphs 2 and 3 of Appendix B of PNAP ADV-36. Source files used: [Source 1] ADV036_Pre-acceptance-requirements of MiC.pdf",4,ADV036_Pre-acceptance-requirements of MiC.pdf,0.579779



✅ Session 10 complete: basic testing finished.


## 9.1 Compare RAG Answers with Reference Answers

Review each RAG answer against the corresponding answer in `RAG Test Questions and Answer.txt`. Semantic similarity is cosine similarity between embeddings from the configured embedding model; use it as a review signal, not as an automatic correctness judgment.

In [18]:
import numpy as np

REFERENCE_ANSWERS = [
    "The Buildings Department will provide a decision within 45 days.",
    "They must clarify or respond to all comments within 30 days from the date of the letter of comments.",
    "The minimum qualification is T3 level, equivalent to a Grade T3 Technically Competent Person as specified in the Code of Practice for Site Supervision. The required supervision frequency is once a week.",
    "The sampling rate must be at least 1% of each type of modular unit delivered to the construction site.",
    "The modules must be manufactured by a factory that holds a valid Full Certificate issued by the Building Technology Research Institute (BTRi) under the MiC Manufacturer Accreditation Scheme, or a factory that holds ISO 9001 or an equivalent quality assurance certification.",
]

if len(test_df) != len(REFERENCE_ANSWERS):
    raise ValueError("Reference answers must match the questions in Section 9.")

answer_texts = test_df["answer"].fillna("").tolist()
all_embeddings = embed_model.get_text_embedding_batch(
    answer_texts + REFERENCE_ANSWERS
)
rag_embeddings = all_embeddings[:len(answer_texts)]
reference_embeddings = all_embeddings[len(answer_texts):]

semantic_similarities = []
for rag_embedding, reference_embedding in zip(rag_embeddings, reference_embeddings):
    denominator = np.linalg.norm(rag_embedding) * np.linalg.norm(reference_embedding)
    similarity = (
        float(np.dot(rag_embedding, reference_embedding) / denominator)
        if denominator
        else np.nan
    )
    semantic_similarities.append(similarity)

answer_review_df = pd.DataFrame(
    {
        "RAG Answer": answer_texts,
        "Reference Answer": REFERENCE_ANSWERS,
        "Semantic Similarity": semantic_similarities,
    },
    index=pd.Index(test_df["question"], name="Question"),
)

display(
    answer_review_df.style
    .format({"Semantic Similarity": "{:.3f}"})
    .set_properties(
        subset=["RAG Answer", "Reference Answer"],
        **{
            "white-space": "pre-wrap",
            "overflow-wrap": "anywhere",
            "min-width": "320px",
            "vertical-align": "top",
        },
    )
)

print("Review each RAG answer against its reference; semantic similarity is a signal, not a correctness verdict.")

2026-09-28 10:11:35,271 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"


,RAG Answer,Reference Answer,Semantic Similarity
Question,,,
How many days does the Buildings Department (BD) take to provide a decision on a pre-submission enquiry?,"Based on the provided context, the Buildings Department (BD) will provide a determination on a pre-submission enquiry within **45 days**. **Source files used:** ADV036_Pre-acceptance-requirements of MiC.pdf",The Buildings Department will provide a decision within 45 days.,0.736
"If the Buildings Department issues a letter of comments on a pre-acceptance application, within how many days must the Authorized Person (AP) and Registered Structural Engineer (RSE) clarify or respond to all comments before the application is considered abandoned?","Based on the provided context, the Authorized Person (AP) and Registered Structural Engineer (RSE) must clarify or respond to all comments within **30 days** from the date of the letter of comments. If they fail to do so, the application will be considered abandoned. Source file(s) used: ADV036_Pre-acceptance-requirements of MiC.pdf",They must clarify or respond to all comments within 30 days from the date of the letter of comments.,0.683
"According to Table 1 of Appendix B, what is the minimum qualification required for a supervisor under the AP Stream, and how frequently must supervision be carried out?","According to Table 1, the minimum qualification required for a supervisor under the AP Stream is Grade T3 (Technically Competent Person), and supervision must be carried out weekly. Source files used: ADV036_Pre-acceptance-requirements of MiC.pdf","The minimum qualification is T3 level, equivalent to a Grade T3 Technically Competent Person as specified in the Code of Practice for Site Supervision. The required supervision frequency is once a week.",0.778
"As an alternative arrangement to quality audits at the prefabrication factory, what is the minimum sampling rate for on-site quality audits of MiC modules delivered to the construction site?","Based on the provided context, the minimum sampling rate for on-site quality audits of MiC modules delivered to the construction site is at least 1% of each type of modular unit delivered. Source files used: ADV036_Pre-acceptance-requirements of MiC.pdf",The sampling rate must be at least 1% of each type of modular unit delivered to the construction site.,0.841
What qualification or certification must a factory have in order to manufacture Modular Integrated Construction (MiC) modules?,"Based on the provided context, a prefabrication factory must hold one of the following qualifications/certifications: 1. A full certificate issued by the Building Technology Research Institute under the **Modular Integrated Construction Manufacturer Accreditation Scheme**; OR 2. **ISO 9001 quality assurance certification** for the prefabrication factory, combined with a Quality Assurance Scheme that complies with paragraphs 2 and 3 of Appendix B of PNAP ADV-36. Source files used: [Source 1] ADV036_Pre-acceptance-requirements of MiC.pdf","The modules must be manufactured by a factory that holds a valid Full Certificate issued by the Building Technology Research Institute (BTRi) under the MiC Manufacturer Accreditation Scheme, or a factory that holds ISO 9001 or an equivalent quality assurance certification.",0.820


Review each RAG answer against its reference; semantic similarity is a signal, not a correctness verdict.


# 10. One-Line RAG with LlamaIndex

We built the pipeline step by step to make each stage visible. LlamaIndex can package the same flow into a query engine:

```python
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=4
)
```

```text
Documents → Nodes → Index → Retriever → Query Engine
```

Understand the pipeline before relying on a convenience API.

In [ ]:
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=TOP_K,
)

framework_response = query_engine.query(
    "What are the minimum supervision frequencies for the AP, RSE and RC streams in Table 1?"
)

print(framework_response)

print("\n✅ Session 15 complete: framework-level RAG executed.")

2026-09-23 16:02:11,601 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/embeddings "HTTP/1.1 200 OK"
2026-09-23 16:02:31,415 - INFO - HTTP Request: POST https://llm-bjb74etky7j3nuou.cn-beijing.maas.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


The minimum supervision frequency for the AP, RSE, and RC streams is weekly.

✅ Session 15 complete: framework-level RAG executed.


# 11. Final Recap

**RAG = retrieve relevant external knowledge → augment the prompt → generate an answer.**

```text
PDF → Load → Chunk → Embed → Store → Retrieve → Augment → Generate
```

## Three Key Ideas

### 1. An LLM is not a knowledge base

The LLM generates answers; the vector store retrieves external knowledge.

### 2. Retrieval is not generation

```text
Retriever: What information should the LLM receive?
LLM:       How should I answer using that information?
```

### 3. RAG quality depends on the whole pipeline

```text
Documents → Chunking → Embeddings → Retrieval → Prompt → Generation
```

RAG engineering is pipeline engineering.

# Appendix A — Environment Checklist

Before running the notebook:

- [ ] Python 3.9 or later and Jupyter installed
- [ ] `DASHSCOPE_API_KEY` configured
- [ ] `DASHSCOPE_BASE_URL` configured or the notebook default selected
- [ ] 3–4 PDFs placed in `data/raw/`
- [ ] Qwen chat and embedding API checks pass

## Appendix B — References

- Alibaba Cloud Model Studio: OpenAI-compatible Chat and Embedding APIs
- LlamaIndex documentation
- Chroma documentation

The provider and endpoint can change without changing the core pipeline.